**ADDESTRAMENTO ENCODER**

SETUP

In [1]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

Progetto: floods


TRAINING

In [2]:
encoders_train_func = project.new_function(
    name="encoders_train-job-v25",
    kind="python",
    python_version="PYTHON3_10",
    code_src="Encoders/", 
    handler="pretrain_encoders", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30"]
)

build = encoders_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")



2026-08-07 11:29:51,923 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run b5614e3124384725904cbe0dd68de31f to finish...
2026-08-07 11:29:57,038 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run b5614e3124384725904cbe0dd68de31f to finish...
2026-08-07 11:30:02,055 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run b5614e3124384725904cbe0dd68de31f to finish...
2026-08-07 11:30:07,070 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run b5614e3124384725904cbe0dd68de31f to finish...
2026-08-07 11:30:12,085 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run b5614e3124384725904cbe0dd68de31f to finish...
2026-08-07 11:30:17,101 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run b5614e3124384725904cbe0dd68de31f to finish...
2026-08-07 11:30:22,119 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run b5614e3124384725904cbe0dd68de31f to finish...
2026-08-07 11

BUILD: COMPLETED


In [3]:
# setup ambiente
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "400Gi"}   
    }
]

parametri = {
    "epochs": 1, 
    "batch_size": 16, 
    "lr": 1e-4, 
    "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                       # sar
    "n_images2": 4, "n_channels2": 10,                      # ottiche               
    "mamba": False, 
    "workers": 0
}

print(f"PARAMETRI: {parametri}")

run_train_encoders = encoders_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xV100",
    local_execution= True,                                # 1x = 1 gpu
    wait=True
)

print(f"STATO FINALE: {run_train_encoders.status.state}")
print(run_train_encoders.logs())

PARAMETRI: {'epochs': 1, 'batch_size': 16, 'lr': 0.0001, 'weight_decay': 0.0001, 'patch_size': 256, 'n_images1': 4, 'n_channels1': 2, 'n_images2': 4, 'n_channels2': 10, 'mamba': False, 'workers': 0}


2026-08-07 11:30:45,234 - dhcore./home/mbarborini/TESI/venv/lib/python3.12/site-packages/digitalhub_runtime_python/runtimes/runtime.py - INFO - Validating task.
2026-08-07 11:30:45,234 - dhcore./home/mbarborini/TESI/venv/lib/python3.12/site-packages/digitalhub_runtime_python/runtimes/runtime.py - INFO - Starting task.
2026-08-07 11:30:45,235 - dhcore./home/mbarborini/TESI/venv/lib/python3.12/site-packages/digitalhub_runtime_python/runtimes/runtime.py - INFO - Configuring execution.
2026-08-07 11:30:45,407 - dhcore./home/mbarborini/TESI/venv/lib/python3.12/site-packages/digitalhub_runtime_python/utils/configuration.py - ERROR - Some error occurred while getting function. Exception: <class 'ModuleNotFoundError'>. Error: ("No module named 'torch'",)
Traceback (most recent call last):
  File "/home/mbarborini/TESI/venv/lib/python3.12/site-packages/digitalhub_runtime_python/utils/configuration.py", line 232, in import_function_from_source
    return _import_function_from_path(function_path,

RuntimeError: Some error occurred while getting function. Exception: <class 'ModuleNotFoundError'>. Error: ("No module named 'torch'",)

RISULTATI

In [ ]:
# salvataggio log
print("SALVATAGGIO METRICHE")
path_s1 = project.get_artifact("metrics-s1").download()
path_s2 = project.get_artifact("metrics-s2").download()

# risultati grafici
df_s1 = pd.read_csv(path_s1)
df_s2 = pd.read_csv(path_s2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# sar
ax1.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
ax1.set_title('Pre-training SAR')
ax1.set_xlabel('Epoche')
ax1.set_ylabel('Loss (MSE)')
ax1.grid(True)

# ottico
ax2.plot(df_s2['epoch'], df_s2['train_loss'], color='red', label='Train Loss')
ax2.set_title('Pre-training OTTICO')
ax2.set_xlabel('Epoche')
ax2.grid(True)

plt.show()